In [5]:
from __future__ import annotations

from importlib.util import find_spec
from pathlib import Path

import pandas as pd

from oracle.models import ModelConfig, Trainer, TrainingConfig
from oracle.utils import load_yaml_config
from oracle.utils.constants import CONFIGS_DIR

In [2]:
training_mapping = load_yaml_config(CONFIGS_DIR / "training.yaml")
model_mapping = load_yaml_config(CONFIGS_DIR / "model.yaml")

shared_mlflow_db = (CONFIGS_DIR.parent / "mlflow.db").resolve()
training_mapping["tracking_uri"] = f"sqlite:///{shared_mlflow_db.as_posix()}"

advanced_models = ["random_forest", "svm_linear", "svm_rbf"]
if find_spec("xgboost") is not None:
    advanced_models.append("xgboost")
if find_spec("lightgbm") is not None:
    advanced_models.append("lightgbm")

print(f"Tracking URI: {training_mapping['tracking_uri']}")
advanced_models

Tracking URI: sqlite:////home/amir/dev/lol-match-oracle/mlflow.db


['random_forest', 'svm_linear', 'svm_rbf', 'xgboost', 'lightgbm']

In [3]:
rows: list[dict[str, object]] = []

for model_name in advanced_models:
    training_config = TrainingConfig.from_mapping(
        training_mapping,
        base_dir=CONFIGS_DIR.parent,
        experiment_name_override="02-advanced-models",
        run_name_override=f"advanced-{model_name}",
    )
    model_config = ModelConfig.from_mapping(
        model_mapping,
        model_name_override=model_name,
    )

    trainer = Trainer(
        training_config=training_config,
        model_config=model_config,
    )

    try:
        result = trainer.train_from_processed_features()
        rows.append(
            {
                "model_name": model_name,
                "run_id": result.run_id,
                **result.metrics,
            }
        )
    except ImportError as exc:
        rows.append(
            {
                "model_name": model_name,
                "status": "skipped",
                "reason": str(exc),
            }
        )

comparison_df = pd.DataFrame(rows)
comparison_df

2026-04-22 19:44:44,474 | INFO | Starting training run 'advanced-random_forest' in experiment '02-advanced-models'
2026-04-22 19:45:06,759 | INFO | Completed training run 'advanced-random_forest' (run_id=43248aab37cc4f57a0825d8c5c25877e)
2026-04-22 19:45:10,006 | INFO | Starting training run 'advanced-svm_linear' in experiment '02-advanced-models'
/home/amir/dev/lol-match-oracle/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
2026-04-22 19:45:41,327 | INFO | Completed training run 'advanced-svm_linear' (run_id=3bfbd859a2274abb8bd0538470cfda70)
2026-04-22 19:45:44,458 | INFO | Starting training run 'advanced-svm_rbf' in experiment '02-advanced-models'
/home/amir/dev/lol-match-oracle/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=3000).  Consider pre-processing

[LightGBM] [Info] Number of positive: 125486, number of negative: 125486
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3930
[LightGBM] [Info] Number of data points in the train set: 250972, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


2026-04-22 19:47:27,732 | INFO | Completed training run 'advanced-lightgbm' (run_id=7c14f1e44f8d4d1e9fefe82d2c650d97)


,model_name,run_id,fit_seconds,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,random_forest,43248aab37cc4f57a0825d8c5c25877e,20.074870,0.988270,0.983339,0.993370,0.988329,0.999408,0.979417,0.972459,0.986780,0.979567,0.998130,0.979584,0.973900,0.985580,0.979706,0.998141
1,svm_linear,3bfbd859a2274abb8bd0538470cfda70,7.924181,0.863909,0.799814,0.970802,0.877051,0.967934,0.861996,0.797943,0.969487,0.875390,0.967692,0.863446,0.799288,0.970631,0.876666,0.967696
2,svm_rbf,0a05e9ec79a84b43b2ce23c92d32be3c,5.928956,0.977914,0.973391,0.982691,0.978019,0.997723,0.976125,0.971705,0.980811,0.976237,0.997472,0.976711,0.972337,0.981341,0.976818,0.997424
3,xgboost,dd5674c248d64908acf971f35dd73314,6.445231,0.987552,0.985718,0.989441,0.987576,0.999334,0.981229,0.978958,0.983600,0.981274,0.998668,0.982164,0.980462,0.983935,0.982195,0.998603
4,lightgbm,7c14f1e44f8d4d1e9fefe82d2c650d97,5.879711,0.985767,0.983686,0.987919,0.985798,0.999290,0.981146,0.979061,0.983321,0.981187,0.998655,0.981843,0.980262,0.983489,0.981873,0.998594


In [4]:
metric_cols = [
    "val_roc_auc",
    "val_f1",
    "test_roc_auc",
    "test_f1",
    "fit_seconds",
]
available_metric_cols = [col for col in metric_cols if col in comparison_df.columns]

comparison_table = comparison_df[["model_name", *available_metric_cols]].copy()
if "val_roc_auc" in comparison_table.columns:
    comparison_table = comparison_table.sort_values("val_roc_auc", ascending=False)

reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
comparison_csv = reports_dir / "advanced_model_comparison_metrics.csv"
comparison_json = reports_dir / "advanced_model_comparison_metrics.json"

comparison_table.to_csv(comparison_csv, index=False)
comparison_table.to_json(comparison_json, orient="records", indent=2)

comparison_table

,model_name,val_roc_auc,val_f1,test_roc_auc,test_f1,fit_seconds
3,xgboost,0.998668,0.981274,0.998603,0.982195,6.445231
4,lightgbm,0.998655,0.981187,0.998594,0.981873,5.879711
0,random_forest,0.998130,0.979567,0.998141,0.979706,20.074870
2,svm_rbf,0.997472,0.976237,0.997424,0.976818,5.928956
1,svm_linear,0.967692,0.875390,0.967696,0.876666,7.924181
